# 04 — Forecasting models

**Phase 4 deliverable:** all 18 upstream architectures scored against both reference lines —
`naive_lag` and `always_long` — on KBANK in one command.

18 notebooks, 4 families. The upstream `deep-learning/` set is one train loop with
`{cell} × {bidirectional} × {paths} × {decoder}` — so it is implemented once and configured 18
times. Same coverage, roughly a fifth of the code, and "compare everything on KBANK" becomes a
single command instead of 18 manual runs.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [2]:
from stock_retrofit.config import all_model_specs
from stock_retrofit.models import registered_kinds

print("model families:", ", ".join(registered_kinds()), "\n")
for spec in all_model_specs():
    print(f"  {spec.name:32s} {spec.kind:28s} <- {spec.upstream}")

model families: always_long, arima, attention, conv, drift, linear, momentum, naive_lag, random_forest, recurrent, seq2seq, stack_encoder_ensemble_xgb, stack_rnn_arima_xgb, xgboost 

  00_always_long                   always_long                  <- (baseline — not an upstream notebook; the reference line for every Sharpe column)
  00_naive_lag                     naive_lag                    <- (baseline — not an upstream notebook; required by spec R8)
  01_lstm                          recurrent                    <- deep-learning/1.lstm.ipynb
  02_bidirectional_lstm            recurrent                    <- deep-learning/2.bidirectional-lstm.ipynb
  03_lstm_2path                    recurrent                    <- deep-learning/3.lstm-2path.ipynb
  04_gru                           recurrent                    <- deep-learning/4.gru.ipynb
  05_bidirectional_gru             recurrent                    <- deep-learning/5.bidirectional-gru.ipynb
  06_gru_2path                     recur

## Run the whole catalogue

Equivalent to:

```bash
python -m stock_retrofit.cli evaluate --all --symbol KBANK
```

Both reference rows are inserted automatically whether or not you ask for them, and pinned to the
top of the table (spec R8): `naive_lag` is the reference for the forecast as a *number*,
`always_long` for the forecast as a *position*.

In [3]:
from stock_retrofit.report import evaluate_symbol

models = evaluate_symbol("KBANK")
models

    00_always_long ... ok
    00_naive_lag ... ok
    01_lstm ... ok
    02_bidirectional_lstm ... ok
    03_lstm_2path ... ok
    04_gru ... ok
    05_bidirectional_gru ... ok
    06_gru_2path ... ok
    07_vanilla ... ok
    08_bidirectional_vanilla ... ok
    09_vanilla_2path ... ok
    10_lstm_seq2seq ... ok
    11_bidirectional_lstm_seq2seq ... ok
    12_lstm_seq2seq_vae ... ok
    13_gru_seq2seq ... ok
    14_bidirectional_gru_seq2seq ... ok
    15_gru_seq2seq_vae ... ok
    16_attention_is_all_you_need ... 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was Tr

ok
    17_cnn_seq2seq ... ok
    18_dilated_cnn_seq2seq ... ok
    19_stack_rnn_arima_xgb ... ok
    20_stack_encoder_ensemble_xgb ... ok
    21_arima ... ok
    22_xgboost ... ok
KBANK — models
                        model symbol  folds   n     ic ic_t   MASE dir_acc RMSE_ret sharpe_net sharpe_gross turnover
                 00_naive_lag  KBANK      8 439      —    — 1.0000       —  0.01347      +0.00        +0.00     0.00
               00_always_long  KBANK      8 439 +0.047 +1.0 1.0010   53.9%  0.01344      +1.69        +1.70     0.00
 14_bidirectional_gru_seq2seq  KBANK      8 439 +0.092 +1.9 1.0017   53.7%  0.01343      +0.95        +1.72     0.13
       19_stack_rnn_arima_xgb  KBANK      8 439 -0.021 -0.4 1.0031   53.4%  0.01344      +1.33        +1.34     0.00
          12_lstm_seq2seq_vae  KBANK      8 439 +0.036 +0.7 1.0059   50.5%  0.01348      +0.66        +0.92     0.05
               13_gru_seq2seq  KBANK      8 439 +0.131 +2.8 1.0088   51.6%  0.01339      +0.83        +

,model,symbol,upstream,folds,n,ic,ic_t,MASE,dir_acc,coverage,flat_share,RMSE_ret,sharpe_net,sharpe_gross,turnover,status
0,00_naive_lag,KBANK,(baseline — not an upstream notebook; required...,8,439,NaN,NaN,1.000000,NaN,0.0,0.129841,0.013471,0.000000,0.000000,0.000000,ok
1,00_always_long,KBANK,(baseline — not an upstream notebook; the refe...,8,439,0.046760,0.978562,1.000967,0.539267,1.0,0.129841,0.013435,1.690536,1.700382,0.002278,ok
2,14_bidirectional_gru_seq2seq,KBANK,deep-learning/14.bidirectional-gru-seq2seq.ipynb,8,439,0.091787,1.926903,1.001727,0.536649,1.0,0.129841,0.013429,0.949896,1.722363,0.127563,ok
3,19_stack_rnn_arima_xgb,KBANK,stacking/stack-rnn-arima-xgb.ipynb,8,439,-0.020832,-0.435570,1.003064,0.534031,1.0,0.129841,0.013442,1.328593,1.339203,0.002278,ok
4,12_lstm_seq2seq_vae,KBANK,deep-learning/12.lstm-seq2seq-vae.ipynb,8,439,0.035670,0.746146,1.005934,0.505236,1.0,0.129841,0.013481,0.664191,0.922648,0.052392,ok
5,13_gru_seq2seq,KBANK,deep-learning/13.gru-seq2seq.ipynb,8,439,0.130995,2.762199,1.008772,0.515707,1.0,0.129841,0.013392,0.825685,1.656010,0.123007,ok
6,11_bidirectional_lstm_seq2seq,KBANK,deep-learning/11.bidirectional-lstm-seq2seq.ipynb,8,439,0.032942,0.689021,1.010145,0.520942,1.0,0.129841,0.013527,0.861141,1.155971,0.061503,ok
7,18_dilated_cnn_seq2seq,KBANK,deep-learning/18.dilated-cnn-seq2seq.ipynb,8,439,0.103086,2.166512,1.010617,0.528796,1.0,0.129841,0.013440,-0.048443,1.884043,0.328018,ok
8,15_gru_seq2seq_vae,KBANK,deep-learning/15.gru-seq2seq-vae.ipynb,8,439,0.039712,0.830825,1.011989,0.515707,1.0,0.129841,0.013527,1.522818,1.934396,0.068337,ok
9,01_lstm,KBANK,deep-learning/1.lstm.ipynb,8,439,0.077590,1.626893,1.014779,0.531414,1.0,0.129841,0.013519,1.324258,1.934026,0.100228,ok


## Reading the result

Read `ic` first — the out-of-sample correlation between forecast and realised return. There is
deliberately **no `beats_naive` column**: MASE < 1.00 on daily returns is a threshold no realistic
forecaster crosses, so a column of `False` would have measured the metric rather than the models.
Check `significant` against `expected_false_positives` below before reading anything into a
2-sigma result — 22 models over one ticker throw one up about half the time, and models sharing a
feature set are not independent draws.

Watch `dir_acc` too. These models sit around 40–45% directional accuracy — *below* a coin flip.
That is not a bug in the harness; it is what happens when a model trained to minimise squared
error on a near-random-walk return series is asked to call direction.

Note also the gap between `sharpe_gross` and `sharpe_net`. A model with a healthy frictionless
Sharpe and a negative net one has found a signal too small to pay for its own turnover — which is
the most common way a backtest lies.

In [6]:
from stock_retrofit.eval import summarise_skill

summary = summarise_skill(models)
print(f"mean IC {summary['mean_ic']:+.3f} over {summary['ran']} models, "
      f"{summary['positive_ic']} positive, {summary['significant']} with |t| > 1.96 "
      f"(~{summary['expected_false_positives']:.0f} expected by chance)")
print(f"{summary['beat_always_long']} of {summary['ran']} models beat holding the share")
print("leaders:", summary["leaders"] or "none")

worst = models.loc[models["status"] == "ok"].nsmallest(5, "sharpe_net")[
    ["model", "MASE", "dir_acc", "sharpe_gross", "sharpe_net", "turnover"]]
print("\nlargest cost drag:")
worst

mean IC +0.041 over 22 models, 17 positive, 2 with |t| > 1.96 (~1 expected by chance)
0 of 22 models beat holding the share
leaders: ['13_gru_seq2seq', '18_dilated_cnn_seq2seq']

largest cost drag:


,model,MASE,dir_acc,sharpe_gross,sharpe_net,turnover
12,21_arima,1.018272,0.481675,1.175204,-2.020834,0.630979
22,20_stack_encoder_ensemble_xgb,1.039528,0.502618,0.946203,-1.193316,0.380410
23,22_xgboost,1.098203,0.513089,1.433236,-0.839964,0.325740
14,07_vanilla,1.019373,0.518325,1.262101,-0.673299,0.350797
7,18_dilated_cnn_seq2seq,1.010617,0.528796,1.884043,-0.048443,0.328018
